# Module 0: Quantum Mechanics Fundamentals & Matrix Mechanics
## Notebook 8: Composite Systems, Tensor Products & Quantum Entanglement

---

### Executive Overview & Theoretical Foundations

Perhaps the most profound departure of quantum physics from classical intuition is **quantum entanglement**, described by Erwin Schrödinger in 1935 as *"the characteristic trait of quantum mechanics, the one that enforces its entire departure from classical lines of thought."*

When two or more quantum systems interact, their state space is not the direct sum of their individual vector spaces, but their **tensor product space**:
$$\mathcal{H}_{AB} = \mathcal{H}_A \otimes \mathcal{H}_B$$
If $\dim(\mathcal{H}_A) = d_A$ and $\dim(\mathcal{H}_B) = d_B$, then $\dim(\mathcal{H}_{AB}) = d_A \cdot d_B$. This exponential scaling of Hilbert space dimension enables both quantum computing speedup and rich many-body correlations.

---

#### 1. Separability vs. Entanglement in Pure States
A pure bipartite state $|\psi_{AB}\rangle \in \mathcal{H}_{AB}$ is **separable (product)** if it can be written as:
$$|\psi_{AB}\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$$
If no such factorization exists, the state is **entangled**. 

##### Partial Trace & Subsystem Entropy
To observe subsystem $A$ alone, one traces out subsystem $B$:
$$\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_{AB}) = \sum_{j} (\hat{I}_A \otimes \langle j|_B) \hat{\rho}_{AB} (\hat{I}_A \otimes |j\rangle_B)$$
For pure states:
- If $|\psi_{AB}\rangle$ is separable $\implies \hat{\rho}_A = |\psi_A\rangle\langle\psi_A|$ is a **pure state** ($\mathrm{Tr}(\hat{\rho}_A^2) = 1$, $S(\hat{\rho}_A) = 0$).
- If $|\psi_{AB}\rangle$ is entangled $\implies \hat{\rho}_A$ is a **mixed state** ($\mathrm{Tr}(\hat{\rho}_A^2) < 1$, $S(\hat{\rho}_A) > 0$).
Entanglement in a pure bipartite state is uniquely quantified by the **von Neumann entropy of the reduced density matrix**:
$$E(|\psi_{AB}\rangle) = S(\hat{\rho}_A) = -\mathrm{Tr}(\hat{\rho}_A \ln \hat{\rho}_A)$$

---

#### 2. The Schmidt Decomposition Theorem
**Theorem**: *Any pure bipartite state $|\psi_{AB}\rangle \in \mathcal{H}_A \otimes \mathcal{H}_B$ can be diagonalized into a single sum of orthonormal product states:*
$$|\psi_{AB}\rangle = \sum_{k=1}^K \sqrt{\lambda_k} |u_k\rangle_A \otimes |v_k\rangle_B$$
where:
- $\lambda_k > 0$ are the **Schmidt coefficients**, satisfying $\sum_{k=1}^K \lambda_k = 1$.
- $\{|u_k\rangle_A\}$ and $\{|v_k\rangle_B\}$ are orthonormal bases for subsystems $A$ and $B$.
- $K \le \min(d_A, d_B)$ is the **Schmidt rank**.

The state is **entangled if and only if $K > 1$**. Furthermore, the eigenvalues of the reduced density matrices $\hat{\rho}_A$ and $\hat{\rho}_B$ are *identical* and equal to the Schmidt weights $\lambda_k$:
$$\hat{\rho}_A = \sum_{k=1}^K \lambda_k |u_k\rangle_A\langle u_k|_A, \quad \hat{\rho}_B = \sum_{k=1}^K \lambda_k |v_k\rangle_B\langle v_k|_B$$
$$S(\hat{\rho}_A) = S(\hat{\rho}_B) = -\sum_{k=1}^K \lambda_k \ln \lambda_k$$

---

#### 3. Mixed State Entanglement: The Peres-Horodecki (PPT) Criterion
For mixed states $\hat{\rho}_{AB} = \sum p_i \hat{\rho}_i^A \otimes \hat{\rho}_i^B$, entropy of reduction is no longer a valid entanglement measure because classical ignorance mixes with quantum correlations.

Instead, we employ the **Peres-Horodecki Positive Partial Transpose (PPT) criterion**:
Taking the partial transpose with respect to subsystem $B$:
$$\langle i_A, j_B | \hat{\rho}_{AB}^{T_B} | k_A, l_B \rangle = \langle i_A, l_B | \hat{\rho}_{AB} | k_A, j_B \rangle$$
- If $\hat{\rho}_{AB}$ is separable, then $\hat{\rho}_{AB}^{T_B} \ge 0$ (all eigenvalues are non-negative).
- **Peres-Horodecki Theorem**: For $2 \times 2$ (two-qubit) and $2 \times 3$ systems, $\hat{\rho}_{AB}$ is **entangled if and only if $\hat{\rho}_{AB}^{T_B}$ has at least one negative eigenvalue**!
The amount of entanglement is quantified by the **Logarithmic Negativity**:
$$E_N(\hat{\rho}) = \log_2 \|\hat{\rho}^{T_B}\|_1 = \log_2\left( 1 + 2 \sum_{\mu_i < 0} |\mu_i| \right)$$

---

### Notebook Implementation Roadmap
1. **Composite Quantum Engine (`CompositeQuantumEngine`)**:
   - Arbitrary tensor products $\hat{A} \otimes \hat{B}$ and partial traces $\mathrm{Tr}_B(\hat{\rho})$.
   - Exact Schmidt decomposition via Singular Value Decomposition (SVD).
   - Partial transpose matrix operations and Logarithmic Negativity.
   - Werner state parameterization $\hat{\rho}_W(p) = p |\Psi^-\rangle\langle\Psi^-| + \frac{1-p}{4}\hat{I}_4$.
2. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: Schmidt coefficient spectra comparing product, non-maximally entangled, and Bell states.
   - *Panel (b)*: Entanglement entropy $S(\hat{\rho}_A)$ and purity $\gamma_A$ vs. superposition angle $\theta$.
   - *Panel (c)*: PPT Peres-Horodecki eigenvalue spectrum vs. Werner parameter $p$, displaying the sharp transition at $p = 1/3$.
   - *Panel (d)*: Density matrix correlation heatmaps showing bipartite quantum coherence.
3. **Rigorous Automated Benchmark Suite**: 5 unit tests validating Schmidt reconstruction, Bell state maximum entanglement, product state zero entropy, Werner state PPT threshold $p=1/3$, and partial trace conservation.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

print(f"NumPy version: {np.__version__}")
print("Composite Systems & Quantum Entanglement environment initialized successfully.")


### Algorithmic Architecture: The CompositeQuantumEngine

The `CompositeQuantumEngine` provides exact linear algebraic implementations:
1. **Partial Trace Algorithm**:
   Reshaping matrix $\hat{\rho}_{AB} \in \mathbb{C}^{d_A d_B \times d_A d_B}$ into a 4-index tensor $\rho_{i_A, j_B, k_A, l_B}$ and contracting over indices $j_B = l_B$:
   $$\hat{\rho}_A[i_A, k_A] = \sum_{j_B=0}^{d_B-1} \rho_{i_A, j_B, k_A, j_B}$$
2. **Schmidt Decomposition via SVD**:
   Reshaping state vector $|\psi_{AB}\rangle \in \mathbb{C}^{d_A d_B}$ into matrix $C_{ij} \in \mathbb{C}^{d_A \times d_B}$.
   Singular Value Decomposition $C = U \Sigma V^\dagger$ yields:
   $$\sqrt{\lambda_k} = \Sigma_{kk}, \quad |u_k\rangle_A = U_{:, k}, \quad |v_k\rangle_B = V_{:, k}^*$$
3. **Partial Transpose**:
   $$\rho^{T_B}_{i_A, j_B, k_A, l_B} = \rho_{i_A, l_B, k_A, j_B}$$
4. **Logarithmic Negativity**:
   $$E_N(\hat{\rho}) = \log_2 \left( \sum_i |\mu_i| \right)$$
   where $\mu_i$ are the eigenvalues of $\hat{\rho}^{T_B}$.


In [ ]:
# Cell 4: Complete Object-Oriented Engine Implementation: CompositeQuantumEngine

class CompositeQuantumEngine:
    '''
    Comprehensive computational suite for Bipartite Composite Systems, Tensor Products,
    Schmidt Decomposition, and Peres-Horodecki (PPT) Entanglement Quantification.
    '''
    
    # Standard Bell States (d_A = d_B = 2)
    BELL_PHI_PLUS = np.array([1.0, 0.0, 0.0, 1.0], dtype=np.complex128) / np.sqrt(2.0)
    BELL_PHI_MINUS = np.array([1.0, 0.0, 0.0, -1.0], dtype=np.complex128) / np.sqrt(2.0)
    BELL_PSI_PLUS = np.array([0.0, 1.0, 1.0, 0.0], dtype=np.complex128) / np.sqrt(2.0)
    BELL_PSI_MINUS = np.array([0.0, 1.0, -1.0, 0.0], dtype=np.complex128) / np.sqrt(2.0)

    @staticmethod
    def partial_trace(rho_AB: np.ndarray, dim_A: int, dim_B: int, trace_out: str = "B") -> np.ndarray:
        '''
        Computes partial trace of bipartite density matrix rho_AB.
        trace_out: "B" returns rho_A (dim_A x dim_A); "A" returns rho_B (dim_B x dim_B).
        '''
        tensor = rho_AB.reshape((dim_A, dim_B, dim_A, dim_B))
        if trace_out.upper() == "B":
            # Contract over axis 1 and 3 (B indices)
            rho_reduced = np.trace(tensor, axis1=1, axis2=3)
        else:
            # Contract over axis 0 and 2 (A indices)
            rho_reduced = np.trace(tensor, axis1=0, axis2=2)
        return 0.5 * (rho_reduced + rho_reduced.T.conj())

    @staticmethod
    def schmidt_decomposition(
        psi_AB: np.ndarray, 
        dim_A: int, 
        dim_B: int
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        '''
        Computes exact Schmidt decomposition of pure bipartite state psi_AB via SVD.
        Returns:
          schmidt_weights: lambda_k (eigenvalues of reduced rho)
          u_vectors: columns are |u_k>_A
          v_vectors: columns are |v_k>_B
        '''
        # Reshape into matrix C such that psi_AB = sum C_{ij} |i>_A |j>_B
        C = psi_AB.reshape((dim_A, dim_B))
        U, s, Vt = la.svd(C, full_matrices=False)
        
        schmidt_weights = s**2
        u_vectors = U
        v_vectors = Vt.T
        
        return schmidt_weights, u_vectors, v_vectors

    @classmethod
    def entanglement_entropy(cls, rho_reduced: np.ndarray) -> float:
        '''
        Computes von Neumann entropy S(rho) = -Tr(rho ln rho).
        '''
        evals = la.eigvalsh(0.5 * (rho_reduced + rho_reduced.T.conj()))
        evals = evals[evals > 1e-15]
        return float(-np.sum(evals * np.log(evals)))

    @staticmethod
    def partial_transpose(rho_AB: np.ndarray, dim_A: int, dim_B: int) -> np.ndarray:
        '''
        Computes partial transpose with respect to subsystem B: rho^{T_B}.
        '''
        tensor = rho_AB.reshape((dim_A, dim_B, dim_A, dim_B))
        # Transpose B indices: axis 1 and axis 3 swapped
        tensor_pt = np.swapaxes(tensor, 1, 3)
        return tensor_pt.reshape((dim_A * dim_B, dim_A * dim_B))

    @classmethod
    def logarithmic_negativity(cls, rho_AB: np.ndarray, dim_A: int, dim_B: int) -> float:
        '''
        Computes Logarithmic Negativity E_N = log2 ||rho^{T_B}||_1.
        '''
        rho_pt = cls.partial_transpose(rho_AB, dim_A, dim_B)
        evals_pt = la.eigvalsh(0.5 * (rho_pt + rho_pt.T.conj()))
        trace_norm = np.sum(np.abs(evals_pt))
        return float(np.log2(trace_norm))

    @classmethod
    def werner_state(cls, p: float) -> np.ndarray:
        '''
        Generates two-qubit Werner state: rho_W(p) = p |Psi-><Psi-| + (1-p)/4 * I_4.
        Entangled if and only if p > 1/3.
        '''
        psi_minus = cls.BELL_PSI_MINUS
        rho_singlet = np.outer(psi_minus, psi_minus.conj())
        rho_werner = p * rho_singlet + (1.0 - p) * 0.25 * np.eye(4, dtype=np.complex128)
        return rho_werner

print("CompositeQuantumEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Grade 4-Panel Visualization Suite

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Composite Quantum Systems, Schmidt Decomposition & Entanglement", fontsize=16, fontweight="bold", y=0.98)

# ------------------------------------------------------------------
# Panel (a): Schmidt Weight Distribution Across State Classes
# ------------------------------------------------------------------
ax_a = axes[0, 0]
# 1. Product state: |00>
psi_prod = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.complex128)
w_prod, _, _ = CompositeQuantumEngine.schmidt_decomposition(psi_prod, 2, 2)

# 2. Partially entangled state: cos(pi/6)|00> + sin(pi/6)|11>
th_partial = np.pi / 6.0
psi_part = np.array([np.cos(th_partial), 0.0, 0.0, np.sin(th_partial)], dtype=np.complex128)
w_part, _, _ = CompositeQuantumEngine.schmidt_decomposition(psi_part, 2, 2)

# 3. Maximally entangled Bell state: (|00> + |11>)/sqrt(2)
w_bell, _, _ = CompositeQuantumEngine.schmidt_decomposition(CompositeQuantumEngine.BELL_PHI_PLUS, 2, 2)

x_bars = np.arange(2)
width = 0.25
ax_a.bar(x_bars - width, w_prod, width, color="#1f77b4", label=r"Product State $|00\rangle$")
ax_a.bar(x_bars, w_part, width, color="#ff7f0e", label=r"Partial $(\theta=\pi/6)$")
ax_a.bar(x_bars + width, w_bell, width, color="#2ca02c", label=r"Bell State $|\Phi^+\rangle$")

ax_a.set_title(r"(a) Schmidt Weights $\lambda_k$ ($|\psi\rangle = \sum \sqrt{\lambda_k}|u_k\rangle|v_k\rangle$)", fontsize=13, fontweight="bold")
ax_a.set_xticks(x_bars)
ax_a.set_xticklabels([r"Mode $k=1$", r"Mode $k=2$"])
ax_a.set_ylabel(r"Schmidt Weight $\lambda_k$")
ax_a.set_ylim(0.0, 1.15)
ax_a.grid(True, linestyle=":", alpha=0.6)
ax_a.legend(loc="upper right", framealpha=0.9)

# ------------------------------------------------------------------
# Panel (b): Entanglement Entropy & Subsystem Purity vs. Theta
# ------------------------------------------------------------------
ax_b = axes[0, 1]
theta_scan = np.linspace(0.0, np.pi/2.0, 120)
entropy_list = []
purity_list = []

for th in theta_scan:
    psi_th = np.array([np.cos(th), 0.0, 0.0, np.sin(th)], dtype=np.complex128)
    rho_full = np.outer(psi_th, psi_th.conj())
    rho_A = CompositeQuantumEngine.partial_trace(rho_full, 2, 2, trace_out="B")
    entropy_list.append(CompositeQuantumEngine.entanglement_entropy(rho_A))
    purity_list.append(np.real(np.trace(rho_A @ rho_A)))

ax_b.plot(theta_scan / np.pi, entropy_list, color="#d62728", lw=2.5, label=r"Entanglement Entropy $S(\hat{\rho}_A)$")
ax_b.plot(theta_scan / np.pi, purity_list, color="#1f77b4", lw=2.2, linestyle="--", label=r"Subsystem Purity $\gamma_A = \mathrm{Tr}(\hat{\rho}_A^2)$")
ax_b.axvline(0.25, color="gray", linestyle=":", lw=1.5, label=r"Max Entanglement $\theta = \pi/4$ ($S=\ln 2$)")

ax_b.set_title(r"(b) Entanglement Metrics for $|\psi(\theta)\rangle = \cos\theta|00\rangle + \sin\theta|11\rangle$", fontsize=13, fontweight="bold")
ax_b.set_xlabel(r"State Angle $\theta / \pi$")
ax_b.set_ylabel(r"Metric Value")
ax_b.set_ylim(0.4, 1.05)
ax_b.grid(True, linestyle=":", alpha=0.6)
ax_b.legend(loc="center right", framealpha=0.9)

# ------------------------------------------------------------------
# Panel (c): PPT Minimum Eigenvalue vs. Werner State Parameter p
# ------------------------------------------------------------------
ax_c = axes[1, 0]
p_scan = np.linspace(0.0, 1.0, 150)
min_eval_list = []
log_neg_list = []

for p_val in p_scan:
    rho_w = CompositeQuantumEngine.werner_state(p_val)
    rho_w_pt = CompositeQuantumEngine.partial_transpose(rho_w, 2, 2)
    evals = la.eigvalsh(0.5 * (rho_w_pt + rho_w_pt.T.conj()))
    min_eval_list.append(np.min(evals))
    log_neg_list.append(CompositeQuantumEngine.logarithmic_negativity(rho_w, 2, 2))

ax_c.plot(p_scan, min_eval_list, color="#9467bd", lw=2.5, label=r"Minimum Eigenvalue $\lambda_{\min}(\hat{\rho}^{T_B})$")
ax_c.plot(p_scan, log_neg_list, color="#2ca02c", lw=2.2, linestyle="--", label=r"Logarithmic Negativity $E_N$ [ebits]")
ax_c.axhline(0.0, color="black", lw=1.0)
ax_c.axvline(1.0/3.0, color="crimson", linestyle=":", lw=2.0, label=r"Peres Threshold $p = 1/3$")

# Shading entangled region
ax_c.axvspan(1.0/3.0, 1.0, color="#2ca02c", alpha=0.12, label="Entangled Domain (PPT Violation)")
ax_c.set_title(r"(c) Peres-Horodecki (PPT) Criterion for Werner States", fontsize=13, fontweight="bold")
ax_c.set_xlabel(r"Singlet Fraction $p$")
ax_c.set_ylabel(r"Eigenvalue / Negativity")
ax_c.set_xlim(0.0, 1.0)
ax_c.grid(True, linestyle=":", alpha=0.6)
ax_c.legend(loc="upper left", framealpha=0.9)

# ------------------------------------------------------------------
# Panel (d): Two-Qubit Density Matrix Coherence Heatmap
# ------------------------------------------------------------------
ax_d = axes[1, 1]
rho_bell = np.outer(CompositeQuantumEngine.BELL_PHI_PLUS, CompositeQuantumEngine.BELL_PHI_PLUS.conj())
im_d = ax_d.imshow(np.real(rho_bell), cmap="Blues", origin="upper")
cbar_d = fig.colorbar(im_d, ax=ax_d, fraction=0.046, pad=0.04)
cbar_d.set_label(r"Real Matrix Element $\mathrm{Re}(\rho_{ij})$", fontsize=11)

basis_labels = [r"$|00\rangle$", r"$|01\rangle$", r"$|10\rangle$", r"$|11\rangle$"]
ax_d.set_xticks(np.arange(4))
ax_d.set_yticks(np.arange(4))
ax_d.set_xticklabels(basis_labels)
ax_d.set_yticklabels(basis_labels)

# Annotate non-zero entries
for i in range(4):
    for j in range(4):
        val = np.real(rho_bell[i, j])
        if abs(val) > 0.01:
            ax_d.text(j, i, f"{val:.2f}", ha="center", va="center", color="white" if val > 0.3 else "black", fontweight="bold")

ax_d.set_title(r"(d) Density Matrix $\hat{\rho}_{AB}$ for Bell State $|\Phi^+\rangle$", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()


### In-Depth Physical Interpretation & Experimental Frontier

#### 1. Schmidt Rank as the Fundamental Entanglement Dimension
The Schmidt decomposition proves that no matter how complex the bipartite Hilbert space $\mathcal{H}_A \otimes \mathcal{H}_B$ is, a pure state can always be expressed in a coordinate system where **each state of system $A$ is paired with exactly one state of system $B$**.
- If $K = 1$, measuring $A$ yields outcome $u_1$ with 100% certainty and leaves $B$ completely unaffected.
- If $K > 1$, observing system $A$ non-locally projects system $B$ into the corresponding paired state $v_k$, demonstrating the quintessential Einstein-Podolsky-Rosen (EPR) steering effect.

#### 2. The Operational Power of the PPT Criterion
In experimental quantum optics (e.g. SPDC sources producing photon pairs), states are unavoidably degraded by dark counts, multi-pair emission, and fiber loss, forming mixed states. 
- While determining whether an arbitrary high-dimensional mixed state is entangled is an **NP-hard problem**, the **Peres-Horodecki PPT criterion** provides an easily computable sufficient condition: simply compute the eigenvalues of $\hat{\rho}^{T_B}$!
- For two qubits (**Panel (c)**), a negative eigenvalue of $\hat{\rho}^{T_B}$ is both **necessary and sufficient** for quantum entanglement, providing an unambiguous test of non-classicality in physical quantum networks.


In [ ]:
# Cell 7: Interactive Exploration & Entanglement Diagnostic Suite

def analyze_bipartite_entanglement(
    theta_superposition: float = np.pi / 4.0, 
    werner_p: float = 0.80
) -> Dict[str, Union[float, bool]]:
    '''
    Evaluates Schmidt decomposition, subsystem entropy, and PPT criteria for pure and mixed states.
    '''
    # 1. Pure state analysis
    psi = np.array([np.cos(theta_superposition), 0.0, 0.0, np.sin(theta_superposition)], dtype=np.complex128)
    weights, _, _ = CompositeQuantumEngine.schmidt_decomposition(psi, 2, 2)
    
    rho_full = np.outer(psi, psi.conj())
    rho_A = CompositeQuantumEngine.partial_trace(rho_full, 2, 2, trace_out="B")
    entropy_A = CompositeQuantumEngine.entanglement_entropy(rho_A)
    purity_A = float(np.real(np.trace(rho_A @ rho_A)))
    
    # 2. Mixed Werner state analysis
    rho_w = CompositeQuantumEngine.werner_state(werner_p)
    rho_w_pt = CompositeQuantumEngine.partial_transpose(rho_w, 2, 2)
    min_eig_pt = float(np.min(la.eigvalsh(0.5 * (rho_w_pt + rho_w_pt.T.conj()))))
    log_neg = CompositeQuantumEngine.logarithmic_negativity(rho_w, 2, 2)
    is_entangled_ppt = min_eig_pt < -1e-12
    
    print("=" * 68)
    print("COMPOSITE QUANTUM SYSTEMS & ENTANGLEMENT DIAGNOSTIC REPORT")
    print("=" * 68)
    print(f"Pure State Angle:                  theta = {theta_superposition / np.pi:.4f} * pi")
    print(f"Schmidt Weights:                   lambda = [{weights[0]:.4f}, {weights[1]:.4f}] (Sum: {sum(weights):.4f})")
    print(f"Subsystem Purity:                  gamma_A = {purity_A:.6f} (Pure=1.0, Mixed < 1.0)")
    print(f"Entanglement Entropy:              S(rho_A) = {entropy_A:.6f} nats (Max: {np.log(2):.6f})")
    print("-" * 68)
    print(f"Werner State Singlet Parameter:    p = {werner_p:.4f} (Threshold: p > 1/3 = 0.3333)")
    print(f"Partial Transpose Min Eigenvalue:  lambda_min(rho^TB) = {min_eig_pt:+.6f}")
    print(f"PPT Entanglement Status:           {'ENTANGLED (PPT Violated)' if is_entangled_ppt else 'SEPARABLE (PPT Satisfied)'}")
    print(f"Logarithmic Negativity:            E_N = {log_neg:.4f} ebits")
    print("=" * 68)
    
    return {
        "weights": weights,
        "purity_A": purity_A,
        "entropy_A": entropy_A,
        "min_eig_pt": min_eig_pt,
        "log_neg": log_neg,
        "is_entangled": is_entangled_ppt
    }

diag_summary = analyze_bipartite_entanglement(theta_superposition=np.pi/4.0, werner_p=0.80)


### Rigorous Automated Verification & Acceptance Suite

The composite quantum engine is subjected to 5 automated unit tests:
1. **Test 1: Schmidt Decomposition State Reconstruction**: Asserts $\|\sum \sqrt{\lambda_k}|u_k\rangle|v_k\rangle - |\psi\rangle\| < 10^{-14}$.
2. **Test 2: Maximally Entangled Bell State Entropy**: Confirms $S(\hat{\rho}_A) = \ln 2 \approx 0.693147$ and $\gamma_A = 0.500000$ to within $10^{-14}$.
3. **Test 3: Separable Product State Zero Entropy**: Asserts $S(\hat{\rho}_A) \equiv 0.000000$ and $\gamma_A \equiv 1.000000$.
4. **Test 4: Peres-Horodecki Werner Threshold**: Verifies that Werner states become entangled precisely at $p > 1/3$ ($0.333333$).
5. **Test 5: Partial Trace Probability Conservation**: Asserts $\mathrm{Tr}(\hat{\rho}_A) = \mathrm{Tr}(\hat{\rho}_{AB}) \equiv 1.00000000$.


In [ ]:
# Cell 9: Automated Benchmark & Acceptance Suite Implementation

def run_benchmark_suite():
    '''
    Executes automated unit tests verifying mathematical and physical accuracy of composite systems.
    '''
    print("=" * 70)
    print("STARTING COMPOSITE SYSTEMS & ENTANGLEMENT BENCHMARK SUITE")
    print("=" * 70)
    
    # ------------------------------------------------------------------
    # Test 1: Schmidt Decomposition Exact Reconstruction
    # ------------------------------------------------------------------
    print("Test 1: Verifying Exact Schmidt Decomposition State Reconstruction...")
    np.random.seed(42)
    psi_rand = np.random.randn(4) + 1j * np.random.randn(4)
    psi_rand /= la.norm(psi_rand)
    
    weights, U, V = CompositeQuantumEngine.schmidt_decomposition(psi_rand, 2, 2)
    psi_reconstructed = np.zeros(4, dtype=np.complex128)
    for k in range(2):
        u_k = U[:, k]
        v_k = V[:, k]
        psi_reconstructed += np.sqrt(weights[k]) * np.kron(u_k, v_k)
        
    recon_err = la.norm(psi_reconstructed - psi_rand)
    print(f"  -> Reconstruction error ||psi_recon - psi||: {recon_err:.2e} (Threshold: < 1e-14)")
    assert recon_err < 1e-14, f"Schmidt reconstruction failed: {recon_err}"
    print("  [PASS] Schmidt decomposition exactly reconstructs bipartite vector.")

    # ------------------------------------------------------------------
    # Test 2: Bell State Maximum Entanglement
    # ------------------------------------------------------------------
    print("\nTest 2: Verifying Maximum Entanglement for Bell State |Phi+>...")
    psi_bell = CompositeQuantumEngine.BELL_PHI_PLUS
    rho_bell = np.outer(psi_bell, psi_bell.conj())
    rho_A = CompositeQuantumEngine.partial_trace(rho_bell, 2, 2, trace_out="B")
    
    entropy_bell = CompositeQuantumEngine.entanglement_entropy(rho_A)
    purity_bell = float(np.real(np.trace(rho_A @ rho_A)))
    target_entropy = np.log(2.0)
    
    print(f"  -> Entropy: {entropy_bell:.8f} (Target: ln 2 = {target_entropy:.8f})")
    print(f"  -> Purity:  {purity_bell:.8f} (Target: 0.50000000)")
    assert abs(entropy_bell - target_entropy) < 1e-14, f"Bell entropy mismatch: {entropy_bell}"
    assert abs(purity_bell - 0.5) < 1e-14, f"Bell purity mismatch: {purity_bell}"
    print("  [PASS] Bell state displays maximal entanglement entropy ln 2.")

    # ------------------------------------------------------------------
    # Test 3: Separable State Zero Entropy
    # ------------------------------------------------------------------
    print("\nTest 3: Verifying Zero Entanglement for Separable State |00>...")
    psi_sep = np.array([1.0, 0.0, 0.0, 0.0], dtype=np.complex128)
    rho_sep = np.outer(psi_sep, psi_sep.conj())
    rho_A_sep = CompositeQuantumEngine.partial_trace(rho_sep, 2, 2, trace_out="B")
    
    entropy_sep = CompositeQuantumEngine.entanglement_entropy(rho_A_sep)
    purity_sep = float(np.real(np.trace(rho_A_sep @ rho_A_sep)))
    
    print(f"  -> Entropy: {entropy_sep:.2e} (Target: 0.00)")
    print(f"  -> Purity:  {purity_sep:.8f} (Target: 1.00000000)")
    assert entropy_sep < 1e-14, f"Separable entropy failed: {entropy_sep}"
    assert abs(purity_sep - 1.0) < 1e-14, f"Separable purity failed: {purity_sep}"
    print("  [PASS] Product state has strictly zero entanglement and unit purity.")

    # ------------------------------------------------------------------
    # Test 4: Peres-Horodecki Threshold for Werner States
    # ------------------------------------------------------------------
    print("\nTest 4: Verifying Peres-Horodecki (PPT) Threshold p = 1/3...")
    # Below threshold p = 0.30 -> separable
    rho_below = CompositeQuantumEngine.werner_state(0.30)
    pt_below = CompositeQuantumEngine.partial_transpose(rho_below, 2, 2)
    min_below = float(np.min(la.eigvalsh(0.5 * (pt_below + pt_below.T.conj()))))
    
    # Above threshold p = 0.40 -> entangled
    rho_above = CompositeQuantumEngine.werner_state(0.40)
    pt_above = CompositeQuantumEngine.partial_transpose(rho_above, 2, 2)
    min_above = float(np.min(la.eigvalsh(0.5 * (pt_above + pt_above.T.conj()))))
    
    print(f"  -> Min eigenvalue at p = 0.30: {min_below:+.6f} (>= 0, Separable)")
    print(f"  -> Min eigenvalue at p = 0.40: {min_above:+.6f} (< 0, Entangled)")
    assert min_below >= -1e-14, f"Werner state below threshold showed negativity: {min_below}"
    assert min_above < 0.0, f"Werner state above threshold failed to show negativity: {min_above}"
    print("  [PASS] PPT criterion strictly isolates the p = 1/3 entanglement boundary.")

    # ------------------------------------------------------------------
    # Test 5: Partial Trace Probability Conservation
    # ------------------------------------------------------------------
    print("\nTest 5: Verifying Partial Trace Normalization Tr(rho_A) == 1.000000...")
    rand_rho = np.random.randn(4, 4) + 1j * np.random.randn(4, 4)
    rand_rho = rand_rho @ rand_rho.T.conj()
    rand_rho /= np.trace(rand_rho)
    
    rho_red_A = CompositeQuantumEngine.partial_trace(rand_rho, 2, 2, trace_out="B")
    tr_A = float(np.real(np.trace(rho_red_A)))
    print(f"  -> Reduced trace: {tr_A:.12f} (Target: 1.000000000000)")
    assert abs(tr_A - 1.0) < 1e-14, f"Partial trace failed to conserve probability: {tr_A}"
    print("  [PASS] Partial trace rigorously preserves total unit probability.")

    print("=" * 70)
    print("ALL 5 COMPOSITE SYSTEMS BENCHMARK TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 70)

run_benchmark_suite()


### Summary of Key Results & Master Mathematical Formulary

| Concept / Quantity | Mathematical Expression | Physical Significance |
| :--- | :--- | :--- |
| **Tensor Product Space** | $\mathcal{H}_{AB} = \mathcal{H}_A \otimes \mathcal{H}_B, \quad \dim = d_A \cdot d_B$ | Exponential state space of composite quantum systems |
| **Partial Trace** | $\hat{\rho}_A = \mathrm{Tr}_B(\hat{\rho}_{AB}) = \sum_j (\hat{I}_A \otimes \langle j|_B) \hat{\rho}_{AB} (\hat{I}_A \otimes |j\rangle_B)$ | Reduction to subsystem observables $\langle\hat{O}_A\rangle = \mathrm{Tr}(\hat{\rho}_A\hat{O}_A)$ |
| **Schmidt Decomposition** | $|\psi_{AB}\rangle = \sum_{k=1}^K \sqrt{\lambda_k}|u_k\rangle_A |v_k\rangle_B$ | Canonical bipartite representation; entangled $\iff K > 1$ |
| **Entanglement Entropy** | $E(|\psi\rangle) = S(\hat{\rho}_A) = -\sum_k \lambda_k \ln \lambda_k$ | Unique entanglement measure for pure bipartite states ($S=\ln 2$ for Bell states) |
| **PPT Criterion (Peres-Horodecki)** | $\hat{\rho}_{AB}^{T_B} \ge 0 \iff \text{Separable (for } 2\times 2, 2\times 3\text{)}$ | Necessary and sufficient test for mixed-state two-qubit entanglement |
| **Logarithmic Negativity** | $E_N(\hat{\rho}) = \log_2 \|\hat{\rho}^{T_B}\|_1 = \log_2(1 + 2\sum_{\mu_i < 0} |\mu_i|)$ | Computable entanglement monotone quantifying distillation bounds |
